# 06 · Rock typing, XRD-calibrated mineral model and rock-type permeability
**F-O gas field — Notebook 06 (completes the Paper 1 workflow; baseline for Paper 3)**

Inputs: unzipped **F-O_composites_v5**; `core_data_shifted.csv` (from F-O_composites_v3); `xrd_F-O6.csv` (14 quantitative XRD
samples, TerraTek 2009, Tables P10–P11) and `core_F-O6` porosity from `core_data_shifted.csv`.

Steps
1. **Core rock types** from Winland r35 (pore-throat radius at 35 % Hg saturation): RT1 ≥ 2 µm … RT4 < 0.1 µm; where they sit
   below the top of the reservoir.
2. **Which logs see which minerals** (XRD vs logs) — PEF is dominated by barite/gypsum mud and is not used.
3. **Mineral model**: clay from an XRD-calibrated linear solver; carbonate cement from the apparent matrix density computed with a
   density-free (neutron–sonic) porosity; quartz+feldspar = remainder; porosity = PHIT (core-calibrated, Notebook 03).
4. **Mineralogy of each rock type** — the diagenetic control on permeability.
5. **Rock type from logs** (random forest, leave-one-well-out) and **rock-type permeability**, tested blind against the field
   transform of Notebook 03.

In [ ]:
# Cell 1 — paths (EDIT)
from pathlib import Path
V5_DIR   = Path(r"C:\Users\YOUR_NAME\...\F-O_composites_v5")
CORE_CSV = Path(r"C:\Users\YOUR_NAME\...\F-O_composites_v3\core_data_shifted.csv")
XRD_CSV  = Path(r"C:\Users\YOUR_NAME\...\FO_petro_v6\xrd_F-O6.csv")
OUT_DIR  = Path(r"C:\Users\YOUR_NAME\...\F-O_v6_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# Cell 2 — load
import numpy as np, pandas as pd, matplotlib.pyplot as plt, lasio, warnings
warnings.filterwarnings("ignore")
from petrotools import mineral as mn, rocktype as rt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
WELLS = ["F-O1", "F-O2", "F-O3", "F-O4", "F-O6", "F-O8", "F-R1", "F-S1"]
WIN = {"F-O1": (3695, 3815.5), "F-O2": (3595, 3699.5), "F-O3": (3678, 3794), "F-O4": (3700, 3876), "F-O6": (3700, 3930),
       "F-O8": (3700, 3916), "F-R1": (3836, 4035), "F-S1": (3596, 3712.5)}
TOP = {"F-O1": 3695.0, "F-O2": 3595.0, "F-O3": 3678.0, "F-O4": 3700.0, "F-O6": 3707.0, "F-S1": 3596.0}   # top reservoir, log depth
logs = {w: pd.read_csv(V5_DIR / f"{w}_composite_v5.csv", index_col=0) for w in WELLS}
core = pd.read_csv(CORE_CSV); xrd = pd.read_csv(XRD_CSV)
core = core[(core.kkl_md > 0.001) & (core.phi_pct > 1)].copy()

## Step 1 — core rock types (Winland r35)

In [ ]:
# Cell 3
core["r35"] = rt.winland_r35(core.kkl_md, core.phi_pct); core["RT"] = rt.rock_type(core.r35.values).values
core["FZI"] = rt.fzi(core.kkl_md, core.phi_pct / 100); core["dz_below_top"] = core.depth_log_m - core.well.map(TOP)
display(pd.crosstab(core.well, core.RT.map(rt.RT_NAMES), normalize="index").round(2))
core.groupby(["well", pd.cut(core.dz_below_top, [-50, 0, 10, 20, 30, 50, 80, 200])]).agg(
    n=("r35", "size"), r35_median_um=("r35", "median"), k_median_mD=("kkl_md", "median")).dropna().round(3)

In [ ]:
# Cell 4 — k-phi with r35 iso-lines
f, a = plt.subplots(figsize=(7, 5.5))
for w, g in core.groupby("well"): a.scatter(g.phi_pct, g.kkl_md, s=7, alpha=0.6, label=w)
x = np.linspace(2, 20, 50)
for r in (0.1, 0.5, 2, 10):
    a.plot(x, 10 ** ((np.log10(r) - 0.732 + 0.864 * np.log10(x)) / 0.588), "k--", lw=0.7); a.text(19.5, 10 ** ((np.log10(r) - 0.732 + 0.864 * np.log10(19.5)) / 0.588), f"{r} µm", fontsize=7)
a.set_yscale("log"); a.set_ylim(1e-3, 1e3); a.set_xlabel("core porosity (%)"); a.set_ylabel("core permeability (mD)"); a.legend(fontsize=7); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_kphi_r35.png", dpi=150); plt.show()

**Reading it.** F-O1 is dominated by RT1 (r35 ≥ 2 µm, median ~40 mD) located 20–50 m below marker M10 — the enhanced-porosity
interval is a distinct unit a little below the top, not the uppermost metres (F-O6's top 10 m are RT4). F-O3 and F-O6 are almost
entirely RT3–RT4. F-O2 and F-O4 are mixed RT2–RT3.

## Step 2 — which logs see which minerals (F-O6 XRD)

In [ ]:
# Cell 5
d6 = logs["F-O6"].copy(); d6["DT"] = d6.DTCO; sh = float(core[core.well == "F-O6"].shift_m.iloc[0])
for c in ["RHOB_USE", "NPHI", "PEF", "DT", "VSH_GR", "POTA", "HTHO"]:
    xrd[c] = [d6[c].loc[z + sh - 0.3: z + sh + 0.3].mean() if c in d6 else np.nan for z in xrd.depth]
c6 = core[core.well == "F-O6"]
xrd["phi_core"] = [c6.phi_pct[(c6.depth_m - z).abs() < 0.35].mean() / 100 if ((c6.depth_m - z).abs() < 0.35).any() else np.nan for z in xrd.depth]
xrd["carb"] = xrd.cal + xrd.sid + xrd.ank + xrd.dol; sand = xrd.depth > 3707
pd.DataFrame({c: [np.corrcoef(xrd.clay[sand], xrd[c][sand])[0, 1], np.corrcoef(xrd.carb[sand], xrd[c][sand])[0, 1]]
              for c in ["RHOB_USE", "NPHI", "PEF", "DT", "VSH_GR", "POTA"]}, index=["r with XRD clay", "r with XRD carbonate"]).round(2)

Clay is seen strongly by K, Th and GR (r ≈ 0.85–0.95); carbonate cement (ankerite/Fe-dolomite + siderite, up to 31 wt%) by
density (+) and sonic (−), but **not by PEF** (a clean quartz sandstone at 3718 m reads PEF ≈ 7: barite/gypsum in the mud).

## Step 3 — mineral model (calibrated to XRD; leave-one-sample-out test)

In [ ]:
# Cell 6
xrd["phi_cal"] = xrd.phi_core.fillna(0.03)                       # mudstones have no plug: 3 % (flagged)
vol = mn.xrd_to_volumes(xrd, xrd.phi_cal); RESP = mn.calibrate(vol, xrd)
nd_resp = {k: v for k, v in RESP.items() if k != "RHOB_USE"}
nd = mn.invert(xrd[["NPHI", "DT", "VSH_GR"]], nd_resp, None); xrd["RHOMAA"] = mn.apparent_matrix_density(xrd.RHOB_USE, nd.PHI_MIN)
CARB = mn.fit_carbonate_index(xrd.RHOMAA[sand], (vol.CARB / (1 - vol.PHI))[sand])
loo = []
for i in range(len(xrd)):
    tr = np.arange(len(xrd)) != i; Ri = mn.calibrate(vol[tr].reset_index(drop=True), xrd[tr].reset_index(drop=True))
    loo.append(mn.invert(xrd.iloc[[i]][["RHOB_USE", "NPHI", "DT", "VSH_GR"]], Ri, None).iloc[0])
loo = pd.DataFrame(loo).reset_index(drop=True)
print("responses:", {t: {g: round(float(v), 3) for g, v in r.items()} for t, r in RESP.items()})
print(f"clay (LOO): r = {np.corrcoef(vol.CLAY, loo.V_CLAY)[0, 1]:.2f}, RMSE = {100 * np.sqrt(np.mean((vol.CLAY - loo.V_CLAY) ** 2)):.1f} vol%")
print(f"carbonate from apparent matrix density (sands): r = {CARB['r']:.2f}, RMSE = {100 * CARB['rmse']:.1f} % of solids")

In [ ]:
# Cell 7 — run the model in every well
MIN = {}
for w in WELLS:
    t, b = WIN[w]; MIN[w] = mn.mineral_model(logs[w].loc[t - 30:b + 30], RESP, CARB)
pd.DataFrame({w: MIN[w].loc[WIN[w][0]:WIN[w][1]][logs[w].loc[WIN[w][0]:WIN[w][1]].VSH_GR < 0.4][["V_CLAY", "V_CARB", "V_QF", "RHOMAA"]].median()
              for w in WELLS}).T.round(3)

**F-O3 carries the most carbonate cement** (apparent matrix density ~2.70) — consistent with its −3 pu density-porosity offset
(Notebook 03), its tight rock and its lack of pay. Caveat: in gas sands the neutron reads low, which biases the density-free
porosity low and RHOMAA low, so carbonate in the best gas sands (F-O1, F-O2) is probably under-estimated.

## Step 4 — mineralogy and log character of each rock type

In [ ]:
# Cell 8
F = ["PHIT", "VSH_GR", "V_CLAY", "V_CARB", "RHOMAA", "NPHI", "DSEP"]
rows = []
for w, g in core.groupby("well"):
    d = logs[w].join(MIN[w][["V_CLAY", "V_CARB", "RHOMAA"]], how="left"); d["DSEP"] = d.NPHI - d.PHIT; z = d.index.to_numpy()
    for _, r in g.iterrows():
        i = int(np.clip(np.searchsorted(z, r.depth_log_m), 1, len(z) - 2)); seg = d.iloc[i - 1:i + 2]
        rows.append({**r.to_dict(), **{f: seg[f].mean() for f in F}, "BADHOLE": seg.BADHOLE.max()})
T = pd.DataFrame(rows); T = T[T.BADHOLE != 1].dropna(subset=F + ["RT"]); T["RT"] = T.RT.astype(int)
T.groupby("RT")[["phi_pct", "kkl_md", "PHIT", "VSH_GR", "V_CLAY", "V_CARB", "RHOMAA", "DSEP"]].median().round(3)

## Step 5 — rock type from logs (blind, leave-one-well-out) and rock-type permeability

In [ ]:
# Cell 9
RF = dict(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=0)
T["lk"] = np.log10(T.kkl_md); T["RT_pred"] = np.nan; perm_rows = []
for w in T.well.unique():
    tr, te = T[T.well != w], T[T.well == w]
    clf = RandomForestClassifier(**RF).fit(tr[F], tr.RT); T.loc[te.index, "RT_pred"] = clf.predict(te[F])
    bF = np.polyfit(100 * tr.PHIT, tr.lk, 1); e_f = np.polyval(bF, 100 * te.PHIT) - te.lk
    tf = rt.rt_transforms(100 * tr.PHIT, tr.lk, tr.RT)
    e_r = np.log10(rt.perm_rt(te.PHIT.to_numpy(), clf.predict_proba(te[F]), clf.classes_, tf)) - te.lk
    perm_rows.append(dict(well=w, n=len(te), field_bias_dex=e_f.mean(), field_rmse_dex=np.sqrt((e_f ** 2).mean()),
                          rocktype_bias_dex=e_r.mean(), rocktype_rmse_dex=np.sqrt((e_r ** 2).mean())))
print(f"rock type, blind: balanced accuracy {balanced_accuracy_score(T.RT, T.RT_pred):.2f} (chance 0.25); "
      f"within one class {(np.abs(T.RT - T.RT_pred) <= 1).mean():.2f}")
display(pd.DataFrame(confusion_matrix(T.RT, T.RT_pred, labels=[1, 2, 3, 4]), index=["RT1", "RT2", "RT3", "RT4"], columns=["pred 1", "pred 2", "pred 3", "pred 4"]))
blind = pd.DataFrame(perm_rows).round(2); blind.to_csv(OUT_DIR / "blind_perm_rocktype_vs_field.csv", index=False); blind

**Result for Paper 1.** With rock types predicted blind from logs, rock-type permeability transforms cut the F-O1 bias from
≈ −1.3 to ≈ −0.3 decades (×19 → ×2 under-prediction) and the mean blind error from ≈ 0.93 to ≈ 0.75 decades. Permeability in
the F-O field is controlled by diagenetic rock type (clay and carbonate cement, pore-throat size), not porosity alone.
The classifier (balanced accuracy ~0.5, 95 % within one class) is the baseline that the ML study (Paper 3) should beat.

In [ ]:
# Cell 10 — final model on all wells, write v6
clf = RandomForestClassifier(**RF).fit(T[F], T.RT); TF = rt.rt_transforms(100 * T.PHIT, T.lk, T.RT)
NEWU = dict(V_QF="V/V", V_CLAY="V/V", V_CARB="V/V", RHOMAA="g/cm3", PHI_ND="V/V", MIN_RESID="", RT_PRED="", RT_P1="", RT_P2="", RT_P3="", RT_P4="", PERM_RT="mD")
for w in WELLS:
    d = logs[w].join(MIN[w], how="left"); d["DSEP"] = d.NPHI - d.PHIT
    ok = d[F].notna().all(axis=1) & (d.index >= WIN[w][0] - 30) & (d.index <= WIN[w][1] + 30)
    for c in ["RT_PRED", "RT_P1", "RT_P2", "RT_P3", "RT_P4", "PERM_RT"]: d[c] = np.nan
    if ok.sum():
        pr = clf.predict_proba(d.loc[ok, F]); d.loc[ok, "RT_PRED"] = clf.classes_[pr.argmax(1)]
        for j, c in enumerate(clf.classes_): d.loc[ok, f"RT_P{int(c)}"] = pr[:, j]
        d.loc[ok, "PERM_RT"] = rt.perm_rt(d.loc[ok, "PHIT"].to_numpy(), pr, clf.classes_, TF)
    d = d.drop(columns=["DSEP"])
    d.to_csv(OUT_DIR / f"{w}_composite_v6.csv", float_format="%.5f")
    units = {c.mnemonic: c.unit for c in lasio.read(str(V5_DIR / f"{w}_composite_v5.las")).curves}; units.update(NEWU)
    las = lasio.LASFile(); las.well["WELL"].value = w; las.well["FLD"].value = "F-O Gas Field"
    las.other = (f"{w} composite v6 (Notebook 06). V_CLAY XRD-calibrated solver; V_CARB from apparent matrix density (neutron-sonic "
                 f"porosity); RT_PRED rock type 1-4 (Winland r35 classes) from random forest; PERM_RT probability-weighted rock-type transform.")
    las.append_curve("DEPT", d.index.to_numpy(), unit="M")
    for c in d.columns: las.append_curve(c, d[c].to_numpy(), unit=units.get(c, ""))
    las.write(str(OUT_DIR / f"{w}_composite_v6.las"), version=2.0, fmt="%.5f")
rows = []
for w in WELLS:
    d = pd.read_csv(OUT_DIR / f"{w}_composite_v6.csv", index_col=0).loc[WIN[w][0]:WIN[w][1]]; s = d[d.VSH_GR < 0.4]
    rows.append(dict(well=w, **{f"frac_RT{k}": round(float((s.RT_PRED == k).mean()), 2) for k in (1, 2, 3, 4)},
                     PERM_RT_median_mD=round(float(s.PERM_RT.median()), 3)))
summary = pd.DataFrame(rows); summary.to_csv(OUT_DIR / "rocktype_summary_v6.csv", index=False); summary

## Notes for the paper
* Rock types are Winland r35 classes; quote r35 thresholds and the RT medians (Cell 8).
* The mineral model is calibrated to 14 XRD samples from **one** well (F-O6); its transfer to other wells assumes similar clay
  and tool responses — neutron tools differ between the 1989–94 SOEKOR wells and the 2009 wells. State this as a limitation and
  recommend XRD on F-O1/F-O2 core (archived) to confirm.
* PEF cannot be used for carbonate (barite/gypsum mud); RHOMAA from a density-free porosity is the carbonate indicator (r 0.73).